
### Data preprocessing

This notebook presents the data cleaning and preprocessing steps applied to the BRFSS dataset.
The goal of this phase is to ensure that the input data is consistent, relevant, and ready for modeling.

Throughout this notebook, we will:

- Inspect and understand the structure of the dataset.

- Handle missing and constant values.

- Detect and treat correlated and constant columns.

- Encode categorical features and normalize continuous ones.

- Prepare the final cleaned dataset ready for model training and evaluation.

Each step is carefully commented to ensure reproducibility and clarity.

In [106]:
import numpy as np
import os
from helpers import load_csv_data

DATA_PATH = "data"

# Useless variables to delete by NAME 
USELESS_VARS = ["FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR","DISPCODE", "SEQNO", "_PSU", "CTELENUM"]



# ---------------- helpers ----------------
def read_header(csv_path):
    """Read first line as header list."""
    with open(csv_path, "r", encoding="utf-8") as f:
        header = f.readline().strip().split(",")
    return [h.strip().strip('"') for h in header]

def header_after_loader(header):
    """helpers.load_csv_data() removes the FIRST column (IDs) from x_* arrays.
    So we drop header[0] to align names with returned matrices."""
    return header[1:]

def indices_for_names(header, names):
    """Return indices in header matching any of names."""
    name_to_idx = {n: i for i, n in enumerate(header)}
    return sorted([name_to_idx[n] for n in names if n in name_to_idx])

def drop_columns_by_idx(X, idxs):
    """Drop columns in X at given indices."""
    if not idxs:
        return X
    keep = [i for i in range(X.shape[1]) if i not in set(idxs)]
    return X[:, keep]


    
def corr_pairwise_nan(X: np.ndarray) -> np.ndarray:
    """" Pairwise Pearson correlation 
    
    For each pair of columns (xi, xj), we:
    1) build a mask m that keeps only rows where both values are finite;
    2) center the two vectors on that overlap: a = xi[m] - mean(xi[m]), b = xj[m] - mean(xj[m]);
    3) compute std(a) and std(b) on the same overlap;
    4) return r_ij = cov(a, b) / (std(a) * std(b)), where cov(a, b) = mean(a * b)."""
    
    n, d = X.shape
    C = np.eye(d, dtype=float)

    for i in range(d):
        xi = X[:, i]
        for j in range(i + 1, d):
            xj = X[:, j]
            m = np.isfinite(xi) & np.isfinite(xj)
            if m.sum() < 2:
                r = 0.0
            else:
                a = xi[m] - np.mean(xi[m])
                b = xj[m] - np.mean(xj[m])
                sa = a.std(ddof=0)
                sb = b.std(ddof=0)
                if sa == 0.0 or sb == 0.0:
                    r = 0.0
                else:
                    r = float(np.mean(a * b) / (sa * sb))
            C[i, j] = C[j, i] = r
    return C


def drop_high_corr(X: np.ndarray, thr: float = 0.95):
    """
Identify and drop highly correlated features using a greedy strategy.

For each pair (i, j) such that |corr(i, j)| >= thr, the feature with the higher index (j)
is marked for removal. This ensures that within every strongly correlated group,
only one representative feature is kept.

Steps:
- Compute the pairwise Pearson correlation matrix (NaN-robust).
- Iterate through all feature pairs (i < j).
- If abs(corr) >= threshold, drop feature j.
- Return the reduced dataset, kept/dropped indices, and the correlation matrix.
"""

    d = X.shape[1]
    if d <= 1:
        return X, np.arange(d), np.array([], dtype=int), [], np.eye(d)

    C = corr_pairwise_nan(X)

    drop_mask = np.zeros(d, dtype=bool)
    corr_pairs = []  

    for i in range(d):
        if drop_mask[i]:
            continue
        for j in range(i + 1, d):
            if drop_mask[j]:
                continue
            corr_val = C[i, j]
            if abs(corr_val) >= thr:
                drop_mask[j] = True
                corr_pairs.append((i, j, corr_val, j)) 

    keep_idx = np.where(~drop_mask)[0]
    drop_idx = np.where(drop_mask)[0]
    X_reduced = X[:, keep_idx]

    return X_reduced, keep_idx, drop_idx, corr_pairs, C


def zscore_fit(X):
    """
Compute column-wise mean and standard deviation ignoring NaN values.
Used for z-score normalization. 
If a column has zero variance (sd = 0), replace it by 1.0 to avoid division by zero.
"""

    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0)
    sd[sd == 0] = 1.0
    return mu, sd

def zscore_apply(X, mu, sd):
    """
Apply z-score normalization using precomputed mean (mu) and std (sd).
Each value is transformed as (x - mu) / sd.
"""
    return (X - mu) / sd

def unique_count_nonan(col):
    """
Count the number of unique non-NaN values in a column.
Used to detect categorical vs continuous features.
"""
    return np.unique(col[~np.isnan(col)]).size

def choose_cat_wide_cols(X):
    """
Heuristic to separate continuous and categorical features.
A column is considered categorical if its number of unique values
is between a lower and upper bound (here 0 and 15).
Returns two lists:
- cont_idx: indices of continuous columns
- cat_idx: indices of categorical columns
"""
    d = X.shape[1]
    cat_idx = []
    cont_idx = []
    for j in range(d):
        k = unique_count_nonan(X[:, j])
        if 0 <= k <= 15:
            cat_idx.append(j)
        else:
            cont_idx.append(j)
    return cont_idx, cat_idx

def fit_one_hot_specs(X, cat_idx):
    """
Build one-hot encoding specifications from the training set.
For each categorical column, store its list of unique (non-NaN) categories.
"""

    specs = {}
    for j in cat_idx:
        vals = np.unique(X[:, j][~np.isnan(X[:, j])])
        specs[j] = vals  
    return specs

def transform_one_hot(X, specs, d_total):
    """
Transform categorical columns into one-hot encoded vectors
based on pre-learned category specs from the training set.
- Each categorical column j is expanded into len(specs[j]) binary columns.
- The last category is dropped to avoid collinearity.
Returns the expanded matrix (np.hstack of all parts).
""" 

    parts = []
    for j in range(d_total):
        col = X[:, j]
        if j in specs:
            cats = specs[j]
            O = np.zeros((X.shape[0], len(cats)))
            for i, v in enumerate(col):
                if not np.isnan(v):
                    pos = np.searchsorted(cats, v)
                    if pos < len(cats) and cats[pos] == v:
                        O[i, pos] = 1.0
            O = O[:, :-1]
            parts.append(O)
        else:
            parts.append(col.reshape(-1, 1))
    return np.hstack(parts)

def fill_nans_with_median(train_data, test_data, cont_idx):
    """
    Fill NaN values in continuous columns with the median value
    learned from the training set.
    """
    train_filled = train_data.copy()
    test_filled = test_data.copy()

    for j in cont_idx:
        col = train_data[:, j]
        median = np.nanmedian(col)
        
        train_filled[np.isnan(train_data[:, j]), j] = median
        test_filled[np.isnan(test_data[:, j]), j] = median

    return train_filled, test_filled


def fill_nans_with_mode(train_data, test_data, cat_idx):
    """
    Fill NaN values in categorical columns with the most frequent (mode) value
    learned from the training set.
    """
    train_filled = train_data.copy()
    test_filled = test_data.copy()

    for j in cat_idx:
        col = train_data[:, j]
        valid = col[~np.isnan(col)]
        
        values, counts = np.unique(valid, return_counts=True)
        mode = values[np.argmax(counts)]

        train_filled[np.isnan(train_data[:, j]), j] = mode
        test_filled[np.isnan(test_data[:, j]), j] = mode

    return train_filled, test_filled

def oversample_minority(X, y, target_ratio, random_state=42):
    """
    Oversample the minority class to reach a target ratio relative to the majority class size.

Steps:
- Identify the minority and majority classes from y.
- Compute how many additional samples are needed to reach 
  target_ratio.
- Randomly sample (with replacement) from the minority indices.
- Append these synthetic samples to X and y (without shuffling).

Used to mitigate class imbalance before model training.
    """
    rng = np.random.default_rng(random_state)

    # Identify minority / majority classes
    labels, counts = np.unique(y, return_counts=True)
    minority_label = labels[np.argmin(counts)]
    majority_label = labels[np.argmax(counts)]

    idx_min = np.where(y == minority_label)[0]
    idx_maj = np.where(y == majority_label)[0]

    n_min = len(idx_min)
    n_maj = len(idx_maj)
    target_min = int(target_ratio * n_maj)

    # Choose random samples (with replacement)
    n_to_add = target_min - n_min
    new_idx = rng.choice(idx_min, size=n_to_add, replace=True)

    # Concatenate new samples after the original data (no shuffle)
    X_res = np.vstack([X, X[new_idx]])
    y_res = np.concatenate([y, y[new_idx]])

    return X_res, y_res



We begin by loading the training and testing datasets using the predefined load_csv_data() function. The headers are read separately and adjusted to remove the first column, which corresponds to the sample IDs. This ensures proper alignment between the headers and the feature matrices.
Certain variables are considered irrelevant or redundant for the predictive task. We identify their indices by name and remove them from both the training and testing sets. The headers are then updated accordingly to maintain correspondence with the cleaned matrices. This reduces the dimensionality of the dataset and eliminates features that could introduce noise or redundancy in subsequent analysis.

In [107]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print(f"[START] x_train: {x_train.shape}")

[START] x_train: (328135, 321)


In [108]:
    # Read headers (align with arrays by removing first column = ID)
h_train = header_after_loader(read_header(os.path.join(DATA_PATH, "x_train.csv")))
h_test  = header_after_loader(read_header(os.path.join(DATA_PATH, "x_test.csv")))
assert len(h_train) == x_train.shape[1], "Header/train mismatch after ID removal"
assert len(h_test)  == x_test.shape[1],  "Header/test mismatch after ID removal"

    # 1) Drop useless vars by NAME on both train and test
drop_idx_train = indices_for_names(h_train, USELESS_VARS)
drop_idx_test  = indices_for_names(h_test,  USELESS_VARS)

x_train_1 = drop_columns_by_idx(x_train, drop_idx_train)
x_test_1  = drop_columns_by_idx(x_test,  drop_idx_test)
    # update headers after drop
h_train = [c for i, c in enumerate(h_train) if i not in set(drop_idx_train)]
h_test  = [c for i, c in enumerate(h_test)  if i not in set(drop_idx_test)]
print(f"[AFTER drop non-relevant vars] x_train: {x_train_1.shape}")

[AFTER drop non-relevant vars] x_train: (328135, 312)


Then, we remove features that contain too many missing values. For each column in the training set, we compute the proportion of missing entries and compare it to a predefined threshold of 80%. Columns exceeding this threshold are considered uninformative and are dropped from both the training and testing datasets. The corresponding headers are also updated to ensure consistent alignment. 

In [109]:
missing_values_threshold = 0.8
columns_to_drop = []
for i in range(x_train_1.shape[1]):
    missing_values = np.isnan(x_train_1[:, i]).sum() / x_train_1.shape[0]
    if missing_values > missing_values_threshold:
            columns_to_drop.append(i)
x_train_cleaned = np.delete(x_train_1, columns_to_drop, axis=1)
x_test_cleaned = np.delete(x_test_1, columns_to_drop, axis=1)
    

# Remove the same columns from the headers
x_train_header_cleaned = [col for i, col in enumerate(h_train) if i not in columns_to_drop]
x_test_header_cleaned = [col for i, col in enumerate(h_test) if i not in columns_to_drop]
    
    
print(f"[AFTER drop rows with >{missing_values_threshold*100}% missing] x_train: {x_train_cleaned.shape}")

[AFTER drop rows with >80.0% missing] x_train: (328135, 196)


At this stage, we identify and remove columns that contain almost identical values across all samples. For each column, we compute the frequency of the most common value (ignoring missing entries) and drop the column if this frequency exceeds 99%. The same indices are removed from both the training and testing datasets, and the corresponding headers are updated. 

In [110]:
CONSTANT_RATIO_THRESHOLD = 0.99

columns_to_drop = []

for i in range(x_train_cleaned.shape[1]):
    col = x_train_cleaned[:, i]
    
    # Ignore NaNs for the check
    valid = col[~np.isnan(col)]

    # Compute frequency of the most common value
    unique_vals, counts = np.unique(valid, return_counts=True)
    most_common_ratio = np.max(counts) / valid.size

    if most_common_ratio >= CONSTANT_RATIO_THRESHOLD:
        columns_to_drop.append(i)

# Apply same drop to both train/test
x_train_cleaned = np.delete(x_train_cleaned, columns_to_drop, axis=1)
x_test_cleaned = np.delete(x_test_cleaned, columns_to_drop, axis=1)

# Clean headers as well
x_train_header_cleaned = [col for i, col in enumerate(x_train_header_cleaned) if i not in columns_to_drop]
x_test_header_cleaned = [col for i, col in enumerate(x_test_header_cleaned) if i not in columns_to_drop]

print(f"[AFTER dropping quasi-constant columns ≥{CONSTANT_RATIO_THRESHOLD*100:.0f}% identical] "
      f"dropped: {len(columns_to_drop)} columns | x_train: {x_train_cleaned.shape}")


[AFTER dropping quasi-constant columns ≥99% identical] dropped: 7 columns | x_train: (328135, 189)


To reduce multicollinearity and prevent redundant information, we remove features that are highly correlated with others. The pairwise Pearson correlation matrix is computed on the training set, and for each pair of features with an absolute correlation above 0.95, only one feature is kept. The same columns are then removed from the test set to maintain alignment. 

In [111]:
x_train_cleaned, keep_idx, drop_idx, corr_pairs, C = drop_high_corr(x_train_cleaned, thr=0.95)

print("Number of Kept indices:", keep_idx.size)
print("Dropped indices:", drop_idx)
print("\nHighly correlated pairs (kept, dropped, corr, dropped_index):")
for kept, dropped, corr_val, dropped_idx in corr_pairs:
    kept_name = x_train_header_cleaned[kept]
    dropped_name = x_train_header_cleaned[dropped]
    print(f"{kept_name} ↔ {dropped_name} | corr = {corr_val:.3f} | dropped: {dropped_name}")

x_test_cleaned = x_test_cleaned[:, keep_idx]
x_train_header_cleaned = [h_train[i] for i in keep_idx]
x_test_header_cleaned  = [h_test[i]  for i in keep_idx]

Number of Kept indices: 158
Dropped indices: [ 31  92  94  95 108 110 114 117 118 119 121 127 129 134 135 144 149 150
 151 152 156 159 160 175 176 177 181 182 183 185 188]

Highly correlated pairs (kept, dropped, corr, dropped_index):
_STATE ↔ _STSTR | corr = 1.000 | dropped: _STSTR
NUMADULT ↔ _RAWRAKE | corr = 0.965 | dropped: _RAWRAKE
CADULT ↔ SEX | corr = 0.985 | dropped: SEX
HAVARTH3 ↔ _DRDXAR1 | corr = 1.000 | dropped: _DRDXAR1
EDUCA ↔ _EDUCAG | corr = 0.980 | dropped: _EDUCAG
SMOKDAY2 ↔ _SMOKER3 | corr = 0.999 | dropped: _SMOKER3
AVEDRNK2 ↔ _DRNKWEK | corr = 0.980 | dropped: _DRNKWEK
AVEDRNK2 ↔ _RFDRHV5 | corr = 0.955 | dropped: _RFDRHV5
EXERANY2 ↔ _TOTINDA | corr = 0.995 | dropped: _TOTINDA
EXERHMM1 ↔ PADUR1_ | corr = 0.996 | dropped: PADUR1_
EXERHMM2 ↔ PADUR2_ | corr = 0.997 | dropped: PADUR2_
LMTJOIN3 ↔ _LMTACT1 | corr = 0.988 | dropped: _LMTACT1
ARTHDIS2 ↔ _LMTWRK1 | corr = 0.987 | dropped: _LMTWRK1
ARTHSOCL ↔ _LMTSCL1 | corr = 0.992 | dropped: _LMTSCL1
HIVTST6 ↔ _AIDTST3 | c

We first distinguish between continuous and categorical features based on the number of unique non-missing values in each column. Features with a limited number of distinct values are treated as categorical, while the others are considered continuous. Two columns: "_STATE" and "EXRACT11" are manually reassigned to the appropriate group to correct misclassifications detected during inspection.
After defining the feature types, missing values are imputed: continuous variables are filled with their median and categorical ones with their mode. Continuous features are then standardized using z-score normalization, where the mean and standard deviation are computed on the training set and consistently applied to the test set. This normalization ensures that all continuous features are on comparable scales, improving model convergence and numerical stability.

In [112]:
cont_idx, cat_idx = choose_cat_wide_cols(x_train_cleaned)
print(f"Chosen {len(cont_idx)} continuous and {len(cat_idx)} categorical (one-hot) columns.")

manual_cats = ["_STATE", "EXRACT11"]
manual_cat_idx = [i for i, name in enumerate(x_train_header_cleaned) if name in manual_cats]

cont_idx = [i for i in cont_idx if i not in manual_cat_idx]

cat_idx = sorted(list(set(cat_idx + manual_cat_idx)))

print(f"Updated feature type assignment:")
print(f"  Continuous: {len(cont_idx)}")
print(f"  Categorical: {len(cat_idx)} (added {manual_cats})")

x_train_cleaned, x_test_cleaned = fill_nans_with_median(x_train_cleaned, x_test_cleaned,cont_idx)
x_train_cleaned, x_test_cleaned = fill_nans_with_mode(x_train_cleaned, x_test_cleaned, cat_idx)

if cont_idx:
    mu, sd = zscore_fit(x_train_cleaned[:, cont_idx])
    x_train_cleaned[:, cont_idx] = zscore_apply(x_train_cleaned[:, cont_idx], mu, sd)
    x_test_cleaned[:,  cont_idx] = zscore_apply(x_test_cleaned[:,  cont_idx],  mu, sd)



Chosen 57 continuous and 101 categorical (one-hot) columns.
Updated feature type assignment:
  Continuous: 55
  Categorical: 103 (added ['_STATE', 'EXRACT11'])


Categorical variables are transformed into binary indicator columns using one-hot encoding. Each categorical feature is expanded into several binary columns, one for each unique category observed in the training data.

In [113]:
specs = fit_one_hot_specs(x_train_cleaned, cat_idx)
x_train_cleaned = transform_one_hot(x_train_cleaned, specs, d_total=len(x_train_header_cleaned))
x_test_cleaned  = transform_one_hot(x_test_cleaned,  specs, d_total=len(x_test_header_cleaned))
print(f"[AFTER scaling + one-hot] x_train: {x_train_cleaned.shape}")


[AFTER scaling + one-hot] x_train: (328135, 527)


To address class imbalance in the training data, the minority class is oversampled to reach 20% of the size of the majority class. This procedure helps the model learn from a more balanced dataset, reducing bias toward the majority class and improving performance on underrepresented outcomes.

In [114]:
x_train_cleaned_up, y_train_up = oversample_minority(x_train_cleaned, y_train, target_ratio=0.2)

print(f"Before: {np.sum(y_train==1)} positive, {np.sum(y_train==-1)} negative")
print(f"After:  {np.sum(y_train_up==1)} positive, {np.sum(y_train_up==-1)} negative")
assert x_train_cleaned_up.shape[0] == y_train_up.shape[0]


Before: 28975 positive, 299160 negative
After:  59832 positive, 299160 negative


After completing all preprocessing steps, the cleaned and balanced datasets are exported to CSV files for future use. The training and testing feature matrices are saved along with their respective headers to preserve variable names.

In [115]:
import csv


with open('data/x_train_cleaned.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(x_train_header_cleaned)
    writer.writerows(x_train_cleaned_up)

with open('data/x_test_cleaned.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(x_test_header_cleaned)
    writer.writerows(x_test_cleaned)


In [116]:
with open('data/y_train_up.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Label"])
    writer.writerows(y_train_up.reshape(-1, 1))